To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your own computer, follow the installation instructions on our Github page [here](https://docs.unsloth.ai/get-started/installing-+-updating).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & [how to save it](#Save)

Visit our docs for all our [model uploads](https://docs.unsloth.ai/get-started/all-our-models) and [notebooks](https://docs.unsloth.ai/get-started/unsloth-notebooks).


### News

**[NEW] We've fixed many bugs in Phi-4** which greatly increases Phi-4's accuracy. See our [blogpost](https://unsloth.ai/blog/phi4)

[NEW] You can view all Phi-4 model uploads with our bug fixes including [dynamic 4-bit quants](https://unsloth.ai/blog/dynamic-4bit), GGUF & more [here](https://huggingface.co/collections/unsloth/phi-4-all-versions-677eecf93784e61afe762afa)

[NEW] As of Novemeber 2024, Unsloth now supports [vision finetuning](https://unsloth.ai/blog/vision)!


### Installation

In [2]:
import torch

print(torch.version.cuda)

12.1


### Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 9000 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = False # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 15 trillion tokens model 2x faster!
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",    # We also uploaded 4bit for 405b!
    "unsloth/Llama-3.3-70B-Instruct-bnb-4bit",
    "unsloth/Mistral-Nemo-Base-2407-bnb-4bit", # New Mistral 12b 2x faster!
    "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
    "unsloth/mistral-7b-v0.3-bnb-4bit",        # Mistral v3 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!
    "unsloth/DeepSeek-R1-Distill-Llama-8B-unsloth-bnb-4bit",
    "unsloth/DeepSeek-R1-Distill-Qwen-14B-unsloth-bnb-4bit",
    "unsloth/DeepSeek-R1-Distill-Qwen-7B-unsloth-bnb-4bit",
] # More models at https://huggingface.co/unsloth

model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-unsloth-bnb-4bit"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

Unsloth: Patching Xformers to fix some performance issues.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 02-16 11:38:35 __init__.py:190] Automatically detected platform cuda.
==((====))==  Unsloth 2025.2.12: Fast Llama patching. Transformers: 4.48.3.
   \\   /|    GPU: Tesla T4. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu121. CUDA: 7.5. CUDA Toolkit: 12.1. Triton: 3.1.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.28.post3. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0.1, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.1.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2025.2.12 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


<a name="Data"></a>
### Data Prep
We now use the Alpaca dataset from [yahma](https://huggingface.co/datasets/yahma/alpaca-cleaned), which is a filtered version of 52K of the original [Alpaca dataset](https://crfm.stanford.edu/2023/03/13/alpaca.html). You can replace this code section with your own data prep.

**[NOTE]** To train only on completions (ignoring the user's input) read TRL's docs [here](https://huggingface.co/docs/trl/sft_trainer#train-on-completions-only).

**[NOTE]** Remember to add the **EOS_TOKEN** to the tokenized output!! Otherwise you'll get infinite generations!

If you want to use the `llama-3` template for ShareGPT datasets, try our conversational [notebook](https://colab.research.google.com/drive/1XamvWYinY6FOSX9GLvnqSjjsNflxdhNc?usp=sharing).

For text completions like novel writing, try this [notebook](https://colab.research.google.com/drive/1ef-tab5bhkvWmBOObepl1WgJvfvSzn5Q?usp=sharing).

In [6]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What's the weather like today?"},
    {"role": "assistant", "content": "The weather is sunny with a chance of rain."},
    {"role": "user", "content": "What about tomorrow?"},
    {"role": "assistant", "content": "The weather is sunny with a chance of rain."},
]
formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False)

print(formatted_prompt)
print("\n")
print(tokenizer.chat_template)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>

What's the weather like today?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

The weather is sunny with a chance of rain.<|eot_id|><|start_header_id|>user<|end_header_id|>

What about tomorrow?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

The weather is sunny with a chance of rain.<|eot_id|>


{{- bos_token }}
{%- if custom_tools is defined %}
    {%- set tools = custom_tools %}
{%- endif %}
{%- if not tools_in_user_message is defined %}
    {%- set tools_in_user_message = true %}
{%- endif %}
{%- if not date_string is defined %}
    {%- set date_string = "26 Jul 2024" %}
{%- endif %}
{%- if not tools is defined %}
    {%- set tools = none %}
{%- endif %}

{#- This block extracts the system message, so we can slot it into the right place. #}
{%- if messages[0]['role

In [ ]:
from elasticsearch import Elasticsearch
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv()

# Replace with your Elasticsearch host and port
ES_HOST = os.getenv("ES_HOST")
ES_USER = os.getenv("ES_USER")
ES_PASSWORD = os.getenv("ES_PASSWORD")
es = Elasticsearch(ES_HOST, basic_auth=(ES_USER, ES_PASSWORD))

# Check connection
if es.ping():
    print("Connected to Elasticsearch")
else:
    print("Failed to connect to Elasticsearch")

response = es.search(
    index="gumball_transcripts_with_prompts",
    query={"match_all": {}},
    size=10000
)

documents = []
for hit in response['hits']['hits']:
    transcript = hit['_source']['transcript']
    
    # Only process if there are at least 2 transcript entries
    if len(transcript) >= 2:
        # Separate the transcript into main part and last 2 entries
        main_transcript = transcript[:-2]
        last_two = transcript[-2:]
        
        # Create the main part of the story
        main_story = "\n".join([
            f"<imagePrompt>\n{item['prompt']}\n</imagePrompt>"
            f"<text>\n{item['text']}\n</text>"
            for item in main_transcript
        ])
        
        # Create the ending part of the story
        ending_story = "\n".join([
            f"<imagePrompt>\n{item['prompt']}\n</imagePrompt>"
            f"<text>\n{item['text']}\n</text>"
            for item in last_two
        ])
        
        document = {
            "instruction": "Generate an story of The Amazing World Of Gumball with the given keywords and size:",
            "input": f"keywords: {hit['_source']['keywords']}; size: {hit['_source']['length']}",
            "output": (
                f"Title: {hit['_source']['title']}\n" +
                main_story + 
                "\n\nPlease finish this Amazing World of Gumball story:\n\n" +
                ending_story
            )
        }
        documents.append(document)

print(f"Loaded {len(documents)} documents")

Connected to Elasticsearch
Loaded 256 documents


In [ ]:
import numpy as np

token_lengths = [len(tokenizer.apply_chat_template([
    {"role": "system", "content": "You are a helpful assistant that generates new transcripts for The Amazing World of Gumball."},
    {"role": "user", "content": f"{doc['instruction']}\n{doc['input']}"},
    {"role": "assistant", "content": doc["output"]}
], tokenize=True)) for doc in documents]

max_length = max(token_lengths)
avg_length = np.mean(token_lengths)

print(f"Max token length: {max_length}")
print(f"Average token length: {avg_length:.2f}")

Max token length: 7980
Average token length: 5686.53


In [ ]:
EOS_TOKEN = tokenizer.eos_token  # Must add EOS_TOKEN

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]
    texts = []

    for instruction, input_text, output in zip(instructions, inputs, outputs):
        # Create message structure for chat template
        messages = [
            {"role": "system", "content": "You are a helpful assistant that generates new transcripts for The Amazing World of Gumball."},
            {"role": "user", "content": f"{instruction}\n{input_text}"},
            {"role": "assistant", "content": output}
        ]

        # Apply chat template
        formatted_text = tokenizer.apply_chat_template(messages, tokenize=False) + EOS_TOKEN
        texts.append(formatted_text)

    return {"text": texts}

from datasets import Dataset

# Load documents into a dataset
dataset = Dataset.from_list(documents)
# Apply formatting function
dataset = dataset.map(formatting_prompts_func, batched=True)
# Shuffle the dataset
dataset = dataset.shuffle(seed=42)  # Optional: Use seed for reproducibility
print(f"Loaded {len(dataset)} documents into the dataset (shuffled)")


# from datasets import load_dataset
# dataset = load_dataset("yahma/alpaca-cleaned", split = "train")
# dataset = dataset.map(formatting_prompts_func, batched = True,)

Map:   0%|          | 0/256 [00:00<?, ? examples/s]

Loaded 256 documents into the dataset (shuffled)


<a name="Train"></a>
### Train the model
Now let's use Huggingface TRL's `SFTTrainer`! More docs here: [TRL SFT docs](https://huggingface.co/docs/trl/sft_trainer). We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`. We also support TRL's `DPOTrainer`!

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 2,
        # warmup_steps = 5,
        warmup_ratio=0.1,
        num_train_epochs = 4, # Set this for 1 full training run.
        # max_steps = 60,
        learning_rate = 3e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
    ),
)

Map (num_proc=2):   0%|          | 0/256 [00:00<?, ? examples/s]

In [11]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.741 GB.
5.516 GB of memory reserved.


In [12]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 256 | Num Epochs = 2
O^O/ \_/ \    Batch size per device = 2 | Gradient Accumulation steps = 2
\        /    Total batch size = 4 | Total steps = 128
 "-____-"     Number of trainable parameters = 41,943,040


Step,Training Loss
1,2.278100
2,2.233400
3,2.220700
4,2.287800
5,2.261700
6,2.160700
7,2.127000
8,2.104600
9,2.056900
10,2.076400


In [13]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

14069.7138 seconds used for training.
234.5 minutes used for training.
Peak reserved memory = 11.662 GB.
Peak reserved memory for training = 6.146 GB.
Peak reserved memory % of max memory = 79.113 %.
Peak reserved memory for training % of max memory = 41.693 %.


<a name="Inference"></a>
### Inference
Let's run the model! You can change the instruction and input - leave the output blank!

**[NEW] Try 2x faster inference in a free Colab for Llama-3.1 8b Instruct [here](https://colab.research.google.com/drive/1T-YBVfnphoVc8E2E854qF3jdia2Ll2W2?usp=sharing)**

In [14]:
# FastLanguageModel.for_inference(model)  # Enable native 2x faster inference

# # Create messages in chat format
# instruction = "Generate an episode of The Amazing World Of Gumball with the given keywords and size:"
# input_text = "keywords: gumball, darwin, school, driving to home, basketball; size: short"
# output = ""
# messages = [
#     {"role": "system", "content": "You are a helpful assistant that generates new transcripts for The Amazing World of Gumball."},
#     {"role": "user", "content": f"{instruction}\n{input_text}"},
# ]

# # Apply chat template
# formatted_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# # Tokenize for model input
# inputs = tokenizer([
#     formatted_input
# ], return_tensors="pt").to("cuda")

# # Generate output
# outputs = model.generate(
#     **inputs,
#     max_new_tokens=1000,
#     use_cache=True,
#     repetition_penalty=1.2,
#     temperature=0.7,
#     top_k=50,
#     top_p=0.95,
#     do_sample=True,
#     early_stopping=True,
# )

# # Decode generated output
# result = tokenizer.batch_decode(outputs)
# print(result)

['<|begin_of_text|><|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 Jul 2024\n\nYou are a helpful assistant that generates new transcripts for The Amazing World of Gumball.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nGenerate an episode of The Amazing World Of Gumball with the given keywords and size:\nkeywords: gumball, darwin, school, driving to home, basketball; size: short<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nTitle: Moving Forward?\nprompt: masterpiece, best quality, 2 figures, boy in t-shirt throwing ball into basket from far away distance, other figure sitting on bench looking worried or concerned expression, dynamic action shot.\n\ntext: Headline: Driving Home From School... With Darwin!\n[The Wattersons\' car is seen leaving Elmore Junior High]\nRichard\'s voice-over as he drives them out of town: I had no idea how much trouble it would be when you kids started coming back here.\nG

In [15]:
# FastLanguageModel.for_inference(model)  # Enable native 2x faster inference

# # Create messages in chat format
# instruction = "Generate an episode of The Amazing World Of Gumball with the given keywords and size:"
# input_text = "keywords: gumball, darwin, school, driving to home, basketball; size: short"
# output = ""
# messages = [
#     {"role": "system", "content": "You are a helpful assistant that generates new transcripts for The Amazing World of Gumball."},
#     {"role": "user", "content": f"{instruction}\n{input_text}"},
#     {"role": "assitant", "content": "Title: Driving Lessons\nprompt: masterpiece, best quality, 1 man wearing casual clothes looking concerned while eating cereal from box in kitchen setting.\n\ntext: Headline: In Bed Eating Cereal!\n[The Episode starts at Elmore Junior High School during lunch time]\nGumball: [Eating] Darwin! I don\'t think this is going too well.\nDarwin: Oh yeah? What do you mean?\n[Gary comes over them carrying two trays full of food on each hand (one tray has no student\'s name)]\nGary: Hey guys what can i get ya today?! You know we got all sorts o\' sandwiches as u like ham cheese turkey or salad...and some fruit if your healthy....um wait were there three students here again??!!\n[Tobias kicks him into his face knocking Gary out cold before he falls down breaking both their tables dropping everything including drinks onto Tobias who screams after tasting something spicy so hot it makes sparks fly around causing fire but only when they\'re near anything flammable then continues screaming until Bobert hits him unconscious also giving up oxygen hence making Darwins eyes turn black due to lack of breathing by doing which saves everyone else except himself)\nHeadline: Lunch Time Again...\n[Cuts back to morning where Richard sits alone outside still inside house though not knowing how its now afternoon because hes been asleep through whole day having eaten breakfast twice already thinking one was dinner!)\nRichard Watterson : Hmm mmm why dont people ever eat proper meals anymore!? Thats probably parta my fault!! Theyre always saying 'hey look ill have mine later' whereas im tryingna make sure kids grow strong enough just remember those words!!!!!!! Mmmmhhhhh.."},
#     {"role": "user", "content": "Can you continue please?"}
# ]

# # Apply chat template
# formatted_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# # Tokenize for model input
# inputs = tokenizer([
#     formatted_input
# ], return_tensors="pt").to("cuda")

# # Generate output
# outputs = model.generate(
#     **inputs,
#     max_new_tokens=1000,
#     use_cache=True,
#     repetition_penalty=1.4,
#     temperature=0.7,
#     top_k=50,
#     top_p=0.95,
#     do_sample=True,
#     early_stopping=True,
# )

# # Decode generated output
# result = tokenizer.batch_decode(outputs)
# print(result)

["<|begin_of_text|><|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 Jul 2024\n\nYou are a helpful assistant that generates new transcripts for The Amazing World of Gumball.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nGenerate an episode of The Amazing World Of Gumball with the given keywords and size:\nkeywords: gumball, darwin, school, driving to home, basketball; size: short<|eot_id|><|start_header_id|>assitant<|end_header_id|>\n\nTitle: Driving Lessons\nprompt: masterpiece, best quality, 1 man wearing casual clothes looking concerned while eating cereal from box in kitchen setting.\n\ntext: Headline: In Bed Eating Cereal!\n[The Episode starts at Elmore Junior High School during lunch time]\nGumball: [Eating] Darwin! I don't think this is going too well.\nDarwin: Oh yeah? What do you mean?\n[Gary comes over them carrying two trays full of food on each hand (one tray has no student's name)]\nGary: Hey guys wh

 You can also use a `TextStreamer` for continuous inference - so you can see the generation token by token, instead of waiting the whole time!

In [ ]:
FastLanguageModel.for_inference(model)  # Enable native 2x faster inference

# Create messages in chat format
instruction = "Generate an episode of The Amazing World Of Gumball with the given keywords and size:"
input_text = "keywords: gumball, darwin, school, driving to home, basketball; size: medium"
output = ""
messages = [
    {"role": "system", "content": "You are a helpful assistant that generates new transcripts for The Amazing World of Gumball."},
    {"role": "user", "content": f"{instruction}\n{input_text}"},
]

# Apply chat template
formatted_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# Tokenize for model input
inputs = tokenizer([
    formatted_input
], return_tensors="pt").to("cuda")

# Stream output during generation
from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(
    **inputs,
    max_new_tokens=max_seq_length,
    use_cache=True,
    repetition_penalty=1.2,
    temperature=0.7,
    top_k=50,
    top_p=0.95,
    do_sample=True,
    early_stopping=True,
    streamer=text_streamer  # Connect streamer
)

<|begin_of_text|><|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a helpful assistant that generates new transcripts for The Amazing World of Gumball.<|eot_id|><|start_header_id|>user<|end_header_id|>

Generate an episode of The Amazing World Of Gumball with the given keywords and size:
keywords: gumball, darwin, school, driving to home, basketball; size: short<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Title: The Drive Home
prompt: masterpiece, best quality, 2 figures, boy in t-shirt, looking at another figure, girl in dress, sad expression, walking away, blurry background, casual setting, concerned expressions.

text: Headline: School Day
[The Wattersons arrive at Elmore Junior High]
Gumball: [To Darwin] I'm so excited! Today's the day we get our licenses!
Darwin: But remember what Principal Brown said.
[Nicole starts singing while opening her trunk full of various items]
Nicole: "Don't yo

KeyboardInterrupt: 

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
from datetime import datetime
import os

# Create main directory if it doesn't exist
os.makedirs("trained_outputs", exist_ok=True)

# Generate a timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Create a subfolder name with timestamp
model_name_clean = model_name.replace("/", "-").replace("\\", "-")
folder_name = f"trained_outputs/{timestamp}_{model_name_clean}"

# Save model and tokenizer locally
model.save_pretrained(folder_name)
tokenizer.save_pretrained(folder_name)

print(f"Model and tokenizer saved to {folder_name}")

'/kaggle/working/20250216_154427_unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit.zip'

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "/kaggle/working/lora_model_20250205_170333", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model) # Enable native 2x faster inference

    # Create messages in chat format
    instruction = "Generate an episode of The Amazing World Of Gumball with the given keywords:"
    input_text = "gumball, darwin, school, driving to home, basketball"
    output = ""
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": f"{instruction}\n{input_text}"},
    ]

    # Apply chat template
    formatted_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # Tokenize for model input
    inputs = tokenizer([
        formatted_input
    ], return_tensors="pt").to("cuda")

    from transformers import TextStreamer
    text_streamer = TextStreamer(tokenizer)
    _ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128)

You can also use Hugging Face's `AutoModelForPeftCausalLM`. Only use this if you do not have `unsloth` installed. It can be hopelessly slow, since `4bit` model downloading is not supported, and Unsloth's **inference is 2x faster**.

In [ ]:
if False:
    # I highly do NOT suggest - use Unsloth if possible
    from peft import AutoPeftModelForCausalLM
    from transformers import AutoTokenizer
    model = AutoPeftModelForCausalLM.from_pretrained(
        "lora_model", # YOUR MODEL YOU USED FOR TRAINING
        load_in_4bit = load_in_4bit,
    )
    tokenizer = AutoTokenizer.from_pretrained("lora_model")

### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_16bit", token = "")

# Merge to 4bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_4bit", token = "")

# Just LoRA adapters
if False: model.save_pretrained_merged("model", tokenizer, save_method = "lora",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "lora", token = "")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [Wiki page](https://github.com/unslothai/unsloth/wiki#gguf-quantization-options)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/drive/1WZDi7APtQ9VsvOrQSSC5DDtxq159j8iZ?usp=sharing)

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("model", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("hf/model", tokenizer, token = "")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "f16", token = "")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "q4_k_m", token = "")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "hf/model", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "",
    )

In [ ]:
print("Done!")

Now, use the `model-unsloth.gguf` file or `model-unsloth-Q4_K_M.gguf` file in llama.cpp or a UI based system like Jan or Open WebUI. You can install Jan [here](https://github.com/janhq/jan) and Open WebUI [here](https://github.com/open-webui/open-webui)

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other links:
1. Llama 3.2 Conversational notebook. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(1B_and_3B)-Conversational.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
6. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://docs.unsloth.ai/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>
